# v13_3 Benchmark Runner — vast.ai RTX Pro 6000

Runs **v13_3 vs v13_2** head-to-head.
New in v13_3: reveal-novelty atoms, reactive-block atoms, sensing prepass rung.

Run cells top-to-bottom once. ls20 uses 1200s/level; ar25 uses 600s/level.

In [ ]:
# 1. Clone / update repo
import os
REPO = "/root/arc3"
BRANCH = "main"
if not os.path.exists(REPO):
    !git clone --depth 1 --branch {BRANCH} https://github.com/shreyasmahimkar/arc-agi-3 {REPO}
else:
    !git -C {REPO} pull --ff-only
print("Repo ready")


In [ ]:
# 2. Install arcengine into the kernel interpreter
import os, sys
WHEELS = "/root/arc3/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
os.system(f"{sys.executable} -m pip install -q --no-index --find-links {WHEELS} arc-agi pydantic python-dotenv")
ret = os.system(f"{sys.executable} -c 'import arcengine; print(\"arcengine OK\")'")
if ret != 0: raise RuntimeError("arcengine import failed — check WHEELS path")


In [ ]:
# 3. Verify hardware
import multiprocessing, platform, subprocess
ncpu = multiprocessing.cpu_count()
print(f"CPUs: {ncpu}  |  OS: {platform.system()}  |  Workers: {ncpu-1}")
try:
    r = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                       capture_output=True, text=True, timeout=5)
    print("GPU:", r.stdout.strip())
except Exception:
    print("No GPU / nvidia-smi not found")


## ls20 — 1200s/level (v13_3 first, then v13_2)

In [ ]:
import os, multiprocessing, subprocess, sys
REPO   = "/root/arc3"
SOLVER = os.path.join(REPO, "CommunitySolutions/chronos_solver/v13_3")
WORKERS = multiprocessing.cpu_count() - 1
print(f"Workers: {WORKERS}")
cmd = [
    sys.executable, "benchmark.py",
    "--games",    "ls20:7",
    "--versions", ".,../v13_2",
    "--budget",   "1200",
    "--workers",  str(WORKERS),
    "--max-states", "10000000",
    "--out",      "ls20_benchmark_1200s.json",
]
subprocess.run(cmd, cwd=SOLVER, env={**os.environ, "PYTHONUNBUFFERED":"1"})


## ar25 — 600s/level

In [ ]:
import os, multiprocessing, subprocess, sys
REPO   = "/root/arc3"
SOLVER = os.path.join(REPO, "CommunitySolutions/chronos_solver/v13_3")
WORKERS = multiprocessing.cpu_count() - 1
cmd = [
    sys.executable, "benchmark.py",
    "--games",    "ar25:3",
    "--versions", ".,../v13_2",
    "--budget",   "600",
    "--workers",  str(WORKERS),
    "--max-states", "10000000",
    "--out",      "ar25_benchmark_600s.json",
]
subprocess.run(cmd, cwd=SOLVER, env={**os.environ, "PYTHONUNBUFFERED":"1"})


## Results

In [ ]:
import json, os
SOLVER = "/root/arc3/CommunitySolutions/chronos_solver/v13_3"
for fname in ["ls20_benchmark_1200s.json", "ar25_benchmark_600s.json"]:
    path = os.path.join(SOLVER, fname)
    if not os.path.exists(path):
        print(f"{fname}: not found yet"); continue
    rows = json.load(open(path))
    print(f"\n=== {fname} ===")
    print(f"  {'Game':<8} {'Level':<6} {'Version':<10} {'Solved':<8} {'Actions':<10} {'Time(s)'}")
    for r in rows:
        solved = "YES" if r.get("solved") else "no"
        print(f"  {r['game']:<8} L{r['level']:<5} {r['version']:<10} {solved:<8} {str(r.get('actions','-')):<10} {round(r.get('elapsed',0),1)}")


In [ ]:
# Print BENCHMARK.md summary
import os
p = "/root/arc3/CommunitySolutions/chronos_solver/v13_3/BENCHMARK.md"
if os.path.exists(p): print(open(p).read())
